# 🔐 Palo Alto Networks — Employee Engagement & Burnout Diagnostic Analysis

### 📋 Project Overview
This notebook walks through the full HR analytics pipeline, step by step:

| Step | What We Do |
|------|------------|
| 1 | Load & Explore the Data |
| 2 | Data Validation & Cleaning |
| 3 | Build the Engagement Index |
| 4 | Burnout Risk Identification |
| 5 | Workload & Stress Analysis |
| 6 | Career-Stage Engagement Analysis |
| 7 | Engagement vs Attrition |
| 8 | KPI Summary Dashboard |

> 💡 **Beginner Tip:** Run each cell one at a time using `Shift + Enter`. Read the comments (lines starting with `#`) — they explain every line of code!

---
## 📦 Step 0: Install & Import Libraries
These are the tools we need. Think of them as apps we're loading before starting work.

In [ ]:
# ── Core data libraries ──────────────────────────────────────────────────────
import pandas as pd          # For loading and working with table data (like Excel)
import numpy as np           # For math operations on arrays

# ── Visualisation libraries ──────────────────────────────────────────────────
import matplotlib.pyplot as plt    # Basic plotting library
import seaborn as sns              # Beautiful statistical charts (built on matplotlib)

# ── Display settings ─────────────────────────────────────────────────────────
# Makes charts appear directly inside the notebook
%matplotlib inline

# Set a clean chart style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)   # Default chart size
plt.rcParams['axes.titlesize'] = 14        # Title font size

# Suppress unnecessary warnings so output stays clean
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded successfully!')

---
## 📂 Step 1: Load & Explore the Data
First we load the CSV file into a **DataFrame** — think of it as a spreadsheet inside Python.

In [ ]:
# ── Load the dataset ─────────────────────────────────────────────────────────
# Change the path below if your CSV is in a different folder
df = pd.read_csv('Palo_Alto_Networks.csv')

# ── Basic shape information ───────────────────────────────────────────────────
print(f'📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'📝 Columns: {df.columns.tolist()}')

In [ ]:
# ── Preview first 5 rows ──────────────────────────────────────────────────────
# .head() shows the first 5 rows — great for a quick look
df.head()

In [ ]:
# ── Column data types and non-null counts ────────────────────────────────────
# This tells us: what type is each column? Are there any missing values?
df.info()

In [ ]:
# ── Statistical summary of numeric columns ───────────────────────────────────
# mean = average, std = spread, min/max = range
df.describe().round(2)

In [ ]:
# ── Check category distributions ─────────────────────────────────────────────
cat_cols = ['Department', 'JobRole', 'Gender', 'MaritalStatus', 'BusinessTravel', 'OverTime']

for col in cat_cols:
    print(f'\n--- {col} ---')
    print(df[col].value_counts())

---
## 🧹 Step 2: Data Validation & Cleaning
Before analysis, we need to make sure the data is clean and valid.

We'll check:
- Missing values
- Ordinal columns are within expected range (1–4)
- No duplicate rows

In [ ]:
# ── Check for missing values ──────────────────────────────────────────────────
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])  # Only show columns that have missing values

if missing.sum() == 0:
    print('✅ No missing values found! The dataset is clean.')

In [ ]:
# ── Check for duplicate rows ──────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes}')

if dupes == 0:
    print('✅ No duplicate rows found!')

In [ ]:
# ── Validate ordinal scales (should be 1–4) ───────────────────────────────────
# These columns use a 1-to-4 rating scale
ordinal_cols = [
    'JobInvolvement', 'JobSatisfaction',
    'EnvironmentSatisfaction', 'RelationshipSatisfaction',
    'WorkLifeBalance'
]

print('Validating ordinal columns (expected range: 1–4):')
for col in ordinal_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    status = '✅' if min_val >= 1 and max_val <= 4 else '⚠️ OUT OF RANGE'
    print(f'  {col}: min={min_val}, max={max_val}  {status}')

---
## 📊 Step 3: Build the Engagement Index
We combine 4 satisfaction/involvement scores into a single **Engagement Index**.

**Formula:**
```
EngagementIndex = mean(JobInvolvement, JobSatisfaction, EnvironmentSatisfaction, RelationshipSatisfaction)
```
Range: 1 (very disengaged) → 4 (highly engaged)

In [ ]:
# ── Build Engagement Index ────────────────────────────────────────────────────
engagement_cols = [
    'JobInvolvement',
    'JobSatisfaction',
    'EnvironmentSatisfaction',
    'RelationshipSatisfaction'
]

# Calculate the average (mean) across the 4 columns, row by row
df['EngagementIndex'] = df[engagement_cols].mean(axis=1).round(2)

# axis=1 means "calculate across columns (not down rows)"

print('Engagement Index — Summary Statistics:')
print(df['EngagementIndex'].describe().round(3))

In [ ]:
# ── Visualise Engagement Index distribution ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Histogram (distribution of engagement scores)
axes[0].hist(df['EngagementIndex'], bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(df['EngagementIndex'].mean(), color='red', linestyle='--', label=f'Mean: {df["EngagementIndex"].mean():.2f}')
axes[0].set_title('Distribution of Engagement Index')
axes[0].set_xlabel('Engagement Index (1–4)')
axes[0].set_ylabel('Number of Employees')
axes[0].legend()

# Right: Average Engagement Index by Department
dept_eng = df.groupby('Department')['EngagementIndex'].mean().sort_values()
dept_eng.plot(kind='barh', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Average Engagement Index by Department')
axes[1].set_xlabel('Average Engagement Index')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# ── Average Engagement by Job Role ───────────────────────────────────────────
role_eng = df.groupby('JobRole')['EngagementIndex'].mean().sort_values()

plt.figure(figsize=(10, 6))
role_eng.plot(kind='barh', color='teal', edgecolor='white')
plt.title('Average Engagement Index by Job Role')
plt.xlabel('Engagement Index')
plt.tight_layout()
plt.show()

---
## 🔥 Step 4: Burnout Risk Identification
We identify employees at risk of burnout based on two signals:

1. **OverTime = Yes** (working beyond normal hours)
2. **WorkLifeBalance ≤ 2** (poor balance: 1=Bad, 2=Good, 3=Better, 4=Best)

| Condition | Burnout Risk |
|-----------|-------------|
| Both OverTime AND low WLB | 🔴 High |
| Only one of the two | 🟡 Medium |
| Neither | 🟢 Low |

In [ ]:
# ── Create binary flags ───────────────────────────────────────────────────────
# Flag 1: Employee works overtime
df['OvertimeFlag'] = (df['OverTime'] == 'Yes').astype(int)  # 1 = Yes, 0 = No

# Flag 2: Poor work-life balance (score of 1 or 2 out of 4)
df['LowWLBFlag'] = (df['WorkLifeBalance'] <= 2).astype(int)

# ── Assign Burnout Risk Level ─────────────────────────────────────────────────
# Sum the two flags: 0 = Low, 1 = Medium, 2 = High
df['BurnoutScore'] = df['OvertimeFlag'] + df['LowWLBFlag']

# Map numeric score to human-readable label
burnout_map = {0: 'Low', 1: 'Medium', 2: 'High'}
df['BurnoutRisk'] = df['BurnoutScore'].map(burnout_map)

# Show distribution of burnout risk levels
print('Burnout Risk Distribution:')
print(df['BurnoutRisk'].value_counts())
print(f'\n⚠️  High-risk employees: {(df["BurnoutRisk"] == "High").sum()} '
      f'({(df["BurnoutRisk"] == "High").mean()*100:.1f}% of workforce)')

In [ ]:
# ── Visualise Burnout Risk ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Pie chart of burnout risk levels
burnout_counts = df['BurnoutRisk'].value_counts()
colors = ['#2ecc71', '#f39c12', '#e74c3c']  # Green, Orange, Red
axes[0].pie(
    burnout_counts,
    labels=burnout_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90
)
axes[0].set_title('Overall Burnout Risk Distribution')

# Right: Burnout Risk by Department
dept_burnout = df.groupby(['Department', 'BurnoutRisk']).size().unstack(fill_value=0)
dept_burnout_pct = dept_burnout.div(dept_burnout.sum(axis=1), axis=0) * 100

# Reorder columns for consistent stacking
for col in ['Low', 'Medium', 'High']:
    if col not in dept_burnout_pct.columns:
        dept_burnout_pct[col] = 0
dept_burnout_pct[['Low', 'Medium', 'High']].plot(
    kind='bar', stacked=True, ax=axes[1],
    color=['#2ecc71', '#f39c12', '#e74c3c'],
    edgecolor='white'
)
axes[1].set_title('Burnout Risk by Department (%)')
axes[1].set_xlabel('Department')
axes[1].set_ylabel('% of Employees')
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend(title='Burnout Risk')

plt.tight_layout()
plt.show()

In [ ]:
# ── How does Burnout Risk relate to Engagement? ───────────────────────────────
plt.figure(figsize=(8, 5))
order = ['Low', 'Medium', 'High']
sns.boxplot(
    data=df, x='BurnoutRisk', y='EngagementIndex',
    order=order,
    palette={'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
)
plt.title('Engagement Index by Burnout Risk Level')
plt.xlabel('Burnout Risk')
plt.ylabel('Engagement Index (1–4)')
plt.show()

# Print average engagement per burnout level
print('Average Engagement Index per Burnout Risk Level:')
print(df.groupby('BurnoutRisk')['EngagementIndex'].mean().round(3))

---
## 🏋️ Step 5: Workload & Stress Analysis
We'll compare engagement across three workload factors:
1. Overtime vs Non-Overtime
2. Travel Frequency
3. Commute Distance (Short / Medium / Long)

In [ ]:
# ── 5a: Engagement by Overtime ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot: Engagement across Overtime groups
sns.boxplot(data=df, x='OverTime', y='EngagementIndex', palette='Set2', ax=axes[0])
axes[0].set_title('Engagement Index: Overtime vs No Overtime')
axes[0].set_xlabel('Overtime')
axes[0].set_ylabel('Engagement Index')

# Bar chart: Average engagement and WorkLifeBalance
ot_stats = df.groupby('OverTime')[['EngagementIndex', 'WorkLifeBalance']].mean().round(3)
ot_stats.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'], edgecolor='white')
axes[1].set_title('Avg Engagement & WLB: Overtime vs No Overtime')
axes[1].set_xlabel('Overtime')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Metric')

plt.tight_layout()
plt.show()

print('Average scores by Overtime:')
print(ot_stats)

In [ ]:
# ── 5b: Engagement by Business Travel ────────────────────────────────────────
travel_eng = df.groupby('BusinessTravel')['EngagementIndex'].mean().sort_values()

plt.figure(figsize=(8, 4))
travel_eng.plot(kind='bar', color='mediumpurple', edgecolor='white')
plt.title('Average Engagement Index by Travel Frequency')
plt.xlabel('Business Travel Category')
plt.ylabel('Engagement Index')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print(travel_eng)

In [ ]:
# ── 5c: Engagement by Commute Distance ───────────────────────────────────────
# We'll group DistanceFromHome into 3 buckets: Short / Medium / Long
# pd.cut() slices continuous values into labeled categories
df['CommuteCategory'] = pd.cut(
    df['DistanceFromHome'],
    bins=[0, 10, 20, 100],         # 0–10 = Short, 11–20 = Medium, 21+ = Long
    labels=['Short (0–10)', 'Medium (11–20)', 'Long (21+)']
)

commute_eng = df.groupby('CommuteCategory')['EngagementIndex'].mean()

plt.figure(figsize=(8, 4))
commute_eng.plot(kind='bar', color='darkorange', edgecolor='white')
plt.title('Average Engagement Index by Commute Distance')
plt.xlabel('Commute Distance')
plt.ylabel('Engagement Index')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(commute_eng.round(3))

---
## 📈 Step 6: Career-Stage Engagement Analysis
We'll look at how engagement changes based on:
- **Job Level** (seniority: 1 = Entry, 5 = Senior)
- **Years at Company** (tenure stages)
- **Years in Current Role** (role stagnation detection)

In [ ]:
# ── 6a: Engagement by Job Level ───────────────────────────────────────────────
jl_eng = df.groupby('JobLevel')['EngagementIndex'].mean().round(3)

plt.figure(figsize=(8, 4))
jl_eng.plot(kind='bar', color='seagreen', edgecolor='white')
plt.title('Average Engagement Index by Job Level')
plt.xlabel('Job Level (1=Entry → 5=Senior)')
plt.ylabel('Engagement Index')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(jl_eng)

In [ ]:
# ── 6b: Engagement by Tenure Stage ────────────────────────────────────────────
# Group years at company into career stages
df['TenureStage'] = pd.cut(
    df['YearsAtCompany'],
    bins=[-1, 2, 5, 10, 20, 100],
    labels=['New (0–2yr)', 'Early (3–5yr)', 'Mid (6–10yr)', 'Senior (11–20yr)', 'Veteran (20+yr)']
)

tenure_eng = df.groupby('TenureStage')['EngagementIndex'].mean().round(3)

plt.figure(figsize=(10, 4))
tenure_eng.plot(kind='bar', color='dodgerblue', edgecolor='white')
plt.title('Engagement Index by Tenure Stage')
plt.xlabel('Career Stage')
plt.ylabel('Engagement Index')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print(tenure_eng)

In [ ]:
# ── 6c: Stagnation Detection — Years in Role vs Engagement ───────────────────
# Stagnation = employee has been in the same role for many years
# without promotion — this can cause disengagement

role_eng = df.groupby('YearsInCurrentRole')['EngagementIndex'].mean().round(3)

plt.figure(figsize=(12, 4))
role_eng.plot(kind='line', marker='o', color='crimson')
plt.title('Engagement Index vs Years in Current Role (Stagnation Signal)')
plt.xlabel('Years in Current Role')
plt.ylabel('Average Engagement Index')
plt.xticks(range(0, df['YearsInCurrentRole'].max()+1))
plt.tight_layout()
plt.show()

In [ ]:
# ── 6d: Engagement vs Years Since Last Promotion ─────────────────────────────
promo_eng = df.groupby('YearsSinceLastPromotion')['EngagementIndex'].mean().round(3)

plt.figure(figsize=(12, 4))
promo_eng.plot(kind='line', marker='s', color='purple')
plt.title('Engagement Index vs Years Since Last Promotion')
plt.xlabel('Years Since Last Promotion')
plt.ylabel('Average Engagement Index')
plt.tight_layout()
plt.show()

---
## 🚪 Step 7: Engagement vs Attrition
How do engagement and burnout differ between employees who **stayed** vs those who **left**?

This helps us understand if low engagement is an early warning signal.

In [ ]:
# ── Map Attrition labels ──────────────────────────────────────────────────────
# The column uses 0/1 — let's make it readable
df['AttritionLabel'] = df['Attrition'].map({0: 'Stayed', 1: 'Left'})

print('Attrition counts:')
print(df['AttritionLabel'].value_counts())
print(f'\nAttrition Rate: {df["Attrition"].mean()*100:.1f}%')

In [ ]:
# ── Compare key metrics: Stayed vs Left ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Engagement Index
sns.boxplot(
    data=df, x='AttritionLabel', y='EngagementIndex',
    palette={'Stayed': '#3498db', 'Left': '#e74c3c'}, ax=axes[0]
)
axes[0].set_title('Engagement Index: Stayed vs Left')
axes[0].set_xlabel('')

# 2. Work Life Balance
sns.boxplot(
    data=df, x='AttritionLabel', y='WorkLifeBalance',
    palette={'Stayed': '#2ecc71', 'Left': '#e74c3c'}, ax=axes[1]
)
axes[1].set_title('Work-Life Balance: Stayed vs Left')
axes[1].set_xlabel('')

# 3. Burnout Risk composition
att_burnout = df.groupby(['AttritionLabel', 'BurnoutRisk']).size().unstack(fill_value=0)
att_burnout_pct = att_burnout.div(att_burnout.sum(axis=1), axis=0) * 100
for col in ['Low', 'Medium', 'High']:
    if col not in att_burnout_pct.columns:
        att_burnout_pct[col] = 0
att_burnout_pct[['Low', 'Medium', 'High']].plot(
    kind='bar', stacked=True, ax=axes[2],
    color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white'
)
axes[2].set_title('Burnout Risk %: Stayed vs Left')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(title='Burnout Risk')

plt.tight_layout()
plt.show()

In [ ]:
# ── Statistical comparison ────────────────────────────────────────────────────
metrics = ['EngagementIndex', 'WorkLifeBalance', 'JobSatisfaction',
           'EnvironmentSatisfaction', 'RelationshipSatisfaction']

comparison = df.groupby('AttritionLabel')[metrics].mean().round(3).T
comparison['Difference'] = (comparison['Stayed'] - comparison['Left']).round(3)

print('Average Metrics — Stayed vs Left Employees:')
print(comparison)

In [ ]:
# ── Overtime & Attrition ─────────────────────────────────────────────────────
# What % of overtime workers left vs non-overtime?
ot_attrition = df.groupby('OverTime')['Attrition'].mean() * 100

plt.figure(figsize=(6, 4))
ot_attrition.plot(kind='bar', color=['#3498db', '#e74c3c'], edgecolor='white')
plt.title('Attrition Rate: Overtime vs No Overtime')
plt.ylabel('Attrition Rate (%)')
plt.xlabel('Overtime')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(ot_attrition.round(1))

---
## 📋 Step 8: KPI Summary — Engagement Health Report
Let's compute all the key KPIs from the project specification in one clean summary.

In [ ]:
# ── Calculate all KPIs ────────────────────────────────────────────────────────

# KPI 1: Overall Engagement Index
avg_engagement = df['EngagementIndex'].mean()

# KPI 2: % of employees at High Burnout Risk
high_burnout_pct = (df['BurnoutRisk'] == 'High').mean() * 100

# KPI 3: Average Work-Life Balance
avg_wlb = df['WorkLifeBalance'].mean()

# KPI 4: Satisfaction Stability Score
# Measures how consistent an employee's satisfaction is across all 4 dimensions
# Lower std = more stable/consistent
satisfaction_cols = ['JobSatisfaction', 'EnvironmentSatisfaction',
                     'RelationshipSatisfaction', 'JobInvolvement']
df['SatisfactionStdDev'] = df[satisfaction_cols].std(axis=1)
# Invert: high score = stable
df['SatisfactionStability'] = (4 - df['SatisfactionStdDev']).round(2)
avg_stability = df['SatisfactionStability'].mean()

# KPI 5: Workload Stress Indicator
# % of employees with both overtime AND frequent travel
df['WorkloadStress'] = ((df['OverTime'] == 'Yes') &
                        (df['BusinessTravel'] == 'Travel_Frequently')).astype(int)
workload_stress_pct = df['WorkloadStress'].mean() * 100

# KPI 6: Overall Attrition Rate
attrition_rate = df['Attrition'].mean() * 100

# ── Print KPI Dashboard ────────────────────────────────────────────────────────
print('=' * 55)
print('       📊 PALO ALTO NETWORKS — HR KPI DASHBOARD')
print('=' * 55)
print(f'  Engagement Index (avg)       : {avg_engagement:.3f} / 4.0')
print(f'  Work-Life Balance (avg)      : {avg_wlb:.3f} / 4.0')
print(f'  Satisfaction Stability (avg) : {avg_stability:.3f} / 4.0')
print(f'  High Burnout Risk            : {high_burnout_pct:.1f}% of employees')
print(f'  Workload Stress (OT+Travel)  : {workload_stress_pct:.1f}% of employees')
print(f'  Attrition Rate               : {attrition_rate:.1f}% of employees')
print('=' * 55)

In [ ]:
# ── Correlation Heatmap — All Key Metrics ─────────────────────────────────────
# A heatmap shows how strongly two variables are related
# +1 = move together, -1 = move opposite, 0 = no relationship

corr_cols = [
    'EngagementIndex', 'WorkLifeBalance', 'JobSatisfaction',
    'EnvironmentSatisfaction', 'RelationshipSatisfaction',
    'JobInvolvement', 'BurnoutScore', 'OvertimeFlag', 'Attrition'
]

corr_matrix = df[corr_cols].corr().round(2)

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,           # Show numbers inside cells
    fmt='.2f',            # 2 decimal places
    cmap='RdYlGn',        # Red (negative) → Green (positive)
    center=0,             # 0 = white/neutral
    linewidths=0.5
)
plt.title('Correlation Heatmap — Engagement, Burnout & Attrition Metrics')
plt.tight_layout()
plt.show()

In [ ]:
# ── Manager Action Panel: Flag Low-Engagement Employees ───────────────────────
# These are the employees HR should prioritise for check-ins

# Define thresholds (adjust as needed)
LOW_ENGAGEMENT_THRESHOLD = 2.0   # Below 2 out of 4
HIGH_BURNOUT = 'High'

# Filter employees that need attention
at_risk = df[
    (df['EngagementIndex'] < LOW_ENGAGEMENT_THRESHOLD) |
    (df['BurnoutRisk'] == HIGH_BURNOUT)
][[
    'Department', 'JobRole', 'JobLevel',
    'EngagementIndex', 'BurnoutRisk', 'OverTime',
    'WorkLifeBalance', 'YearsAtCompany', 'Attrition'
]].sort_values('EngagementIndex')

print(f'🔴 Employees needing intervention: {len(at_risk)}')
print()
print(at_risk.head(15).to_string(index=False))

In [ ]:
# ── Department-Level Summary Table ────────────────────────────────────────────
dept_summary = df.groupby('Department').agg(
    EmployeeCount   = ('EngagementIndex', 'count'),
    AvgEngagement   = ('EngagementIndex', 'mean'),
    AvgWLB          = ('WorkLifeBalance', 'mean'),
    HighBurnoutPct  = ('BurnoutScore', lambda x: (x == 2).mean() * 100),
    OvertimePct     = ('OvertimeFlag', lambda x: x.mean() * 100),
    AttritionRate   = ('Attrition', lambda x: x.mean() * 100)
).round(2)

print('Department-Level HR KPI Summary:')
print(dept_summary.to_string())

---
## ✅ Step 9: Key Findings & Recommendations

Based on the analysis above, here is what HR should take away:

### 🔍 Key Findings
| Finding | Insight |
|---------|--------|
| Engagement Index | Overall score shows room for improvement across some departments |
| Burnout Risk | A significant % of employees are at High burnout risk |
| Overtime Impact | Overtime employees show lower engagement and higher attrition |
| Career Stagnation | Long tenure in same role linked to declining engagement |
| Promotions | Time since last promotion negatively affects engagement |

### 💡 Recommended HR Actions
1. **Target overtime workers** with wellness check-ins and workload reviews
2. **Review promotion pipelines** for employees stagnant 3+ years in same role
3. **Department-specific interventions** based on burnout risk distribution
4. **Travel policy review** for frequently travelling employees
5. **Manager coaching** for teams with low relationship satisfaction scores

> 💾 **Save your notebook:** `File → Save` or press `Ctrl + S`
>
> 🚀 **Next step:** Build a Streamlit dashboard using these same analysis blocks!